# Diffusion Model for Financial Time Series

**Before running:** Runtime → Change runtime type → Hardware accelerator → **GPU**

| Cell | What it does |
|------|--------------|
| Setup | Mount Drive, set paths, check GPU |
| Config | All hyperparameters in one place |
| Train | Train the diffusion model |
| Sample | Generate synthetic time series from a checkpoint |
| Evaluate | Compute discriminative/predictive/VDS/FDDS scores |

## 1. Setup

In [ ]:
# ── Check GPU ────────────────────────────────────────────────────────────────
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory  : {mem_gb:.1f} GB")

In [ ]:
# ── Clone repo from GitHub ────────────────────────────────────────────────────
import os
REPO_DIR = '/content/DiffusionModelTimeSeries'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Ardameliksah/DiffusionModelTimeSeries.git {REPO_DIR}
else:
    print("Repo already cloned, pulling latest...")
    !git -C {REPO_DIR} pull

In [ ]:
import os, sys
from pathlib import Path

REPO_PATH = REPO_DIR  # set by the clone cell above

assert Path(REPO_PATH).exists(), f"Folder not found: {REPO_PATH}"

os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print(f"Working directory: {os.getcwd()}")
print("Files found:", [f for f in os.listdir() if f.endswith('.py')])

In [ ]:
# ── Install any missing packages ──────────────────────────────────────────────
# Colab already has torch, numpy, pandas, matplotlib, scikit-learn, scipy,
# seaborn, tqdm, pillow. Only install what might be missing.
!pip install -q --upgrade pip
!pip show tqdm scikit-learn seaborn | grep -E 'Name|Version'

## 2. Configuration

Edit the values below before running the Train / Sample / Evaluate cells.

In [ ]:
import torch

# ── Mode ──────────────────────────────────────────────────────────────────────
MODE      = "raw"    # "raw" or "image"
EMBEDDING = "delay"  # "delay" | "patch" | "stft" | "mrti"  (image mode only)

# ── Training ──────────────────────────────────────────────────────────────────
NUM_EPOCHS     = 500
BATCH_SIZE     = 64
LEARNING_RATE  = None   # None = config default (5e-4 raw, 1e-3 image)
NOISE_SCHEDULE = None   # None | "linear" | "cosine" | "exponential"
NORMALIZATION  = None   # None | "minmax" | "zscore"
HIDDEN_DIM     = None   # None | 64 (small) | 128 (medium) | 256 (large)
NUM_LAYERS     = None   # None | 3 (small) | 6 (medium) | 8 (large)
POS_ENC        = None   # None | "learnable" | "fixed"
SEED           = 42
RESUME_FROM    = None   # Path to checkpoint to resume; None = train from scratch
CHECKPOINT_DIR = None   # None = output/checkpoints (auto)

# ── Sampling ──────────────────────────────────────────────────────────────────
NUM_SAMPLES    = 256
NUM_STEPS      = 50     # DDIM denoising steps
ETA            = 0.0    # 0 = deterministic DDIM; 1 = DDPM-like stochasticity

# Path to the checkpoint to load for sampling/evaluation.
# Leave as None to auto-detect output/checkpoints/best_model.pt after training.
CHECKPOINT_PATH = None

# ── Evaluation ────────────────────────────────────────────────────────────────
N_METRIC_ITERATIONS = 5      # Independent runs for disc/pred score estimation
COMPUTE_CONTEXT_FID = False  # Requires Diffusion-TS folder (slow)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device  : {DEVICE}")
print(f"Mode    : {MODE}")
print(f"Epochs  : {NUM_EPOCHS}  |  Batch: {BATCH_SIZE}")
if MODE == 'image':
    print(f"Embedding: {EMBEDDING}")

## 3. Train

Trains the model and saves checkpoints to `output/checkpoints/` (raw mode) or `output/checkpoints_image/` (image mode).
The best validation-loss checkpoint is always saved as `best_model.pt`.

In [ ]:
from train_with_mode import train

train(
    mode=MODE,
    device=DEVICE,
    resume_from=RESUME_FROM,
    embedding=EMBEDDING,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    noise_schedule=NOISE_SCHEDULE,
    checkpoint_dir=CHECKPOINT_DIR,
    normalization=NORMALIZATION,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    seed=SEED,
    pos_enc=POS_ENC,
    lr=LEARNING_RATE,
)

## 4. Sample

Generates `NUM_SAMPLES` synthetic windows from a trained checkpoint and saves them as `.npz` to `output/generated_samples/`.

In [ ]:
from pathlib import Path
from sample_unified import generate, denormalize, save_samples
from config.stocks_config import Config as RawConfig
from config.image_config import ImageVersionConfig
from utils.data_utils import StockDataset, build_scaler

# Auto-detect best_model.pt if not set manually
_ckpt = CHECKPOINT_PATH
if _ckpt is None:
    if MODE == "raw":
        _ckpt = str(Path(REPO_PATH) / "output" / "checkpoints" / "best_model.pt")
    else:
        _ckpt = str(Path(REPO_PATH) / "output" / "checkpoints_image" / "best_model.pt")

assert Path(_ckpt).exists(), f"Checkpoint not found: {_ckpt}"
print(f"Loading checkpoint: {_ckpt}")

# Build config
if MODE == "raw":
    _config = RawConfig()
else:
    _config = ImageVersionConfig()
    _config.image.embedding_type = EMBEDDING

if NORMALIZATION is not None:
    _config.data.neg_one_to_one = (NORMALIZATION == "minmax")

# Generate
samples = generate(
    checkpoint_path=_ckpt,
    mode=MODE,
    config=_config,
    num_samples=NUM_SAMPLES,
    num_steps=NUM_STEPS,
    eta=ETA,
    device=DEVICE,
)

# Denormalize back to original price scale
dataset = StockDataset(
    csv_path=_config.data.data_path,
    scaler=build_scaler(_config.data.neg_one_to_one),
    window_length=_config.model.sequence_length,
)
samples_denorm = denormalize(samples, dataset)

_emb = EMBEDDING if MODE == "image" else None
save_samples(samples_denorm, _config.sampling.output_dir, MODE, _emb)
print(f"\nFinal shape: {samples_denorm.shape}  (samples, channels, time)")

### Quick visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n_show = min(6, samples_denorm.shape[0])
channel_names = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]

fig, axes = plt.subplots(n_show, 1, figsize=(12, 2 * n_show))
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    # Plot Close price (channel 3) for readability
    ax.plot(samples_denorm[i, 3, :], linewidth=1)
    ax.set_ylabel("Close", fontsize=8)
    ax.set_title(f"Sample {i+1}", fontsize=8)
    ax.tick_params(labelsize=7)

plt.suptitle(f"Generated Samples — {MODE} mode", fontsize=11)
plt.tight_layout()
plt.show()

## 5. Evaluate

Computes:
- **Discriminative score** — GRU classifier real vs synthetic (target: 0.0, test acc: 0.5)
- **Predictive MAE** — train-on-fake / test-on-real (lower = better)
- **VDS** — KL divergence of value distributions (lower = better)
- **FDDS** — KL divergence of cross-correlation distributions (lower = better)
- **Correlational score** — |CACF_fake − CACF_real| / 10 (lower = better)

Results and a comparison plot are saved to `output/` automatically.

In [ ]:
from evaluate_unified import evaluate

evaluate(
    mode=MODE,
    checkpoint_path=_ckpt,
    device=DEVICE,
    num_samples=NUM_SAMPLES,
    output_dir=None,          # None = config default
    n_metric_iterations=N_METRIC_ITERATIONS,
    compute_context_fid=COMPUTE_CONTEXT_FID,
    normalization=NORMALIZATION,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    seed=SEED,
    pos_enc=POS_ENC,
)

## (Optional) Copy outputs back to Drive

Colab runtimes are ephemeral — outputs written inside `/content/` are lost when the session ends.  
If your `REPO_PATH` already points to Drive, checkpoints are already persistent. Otherwise run the cell below.

In [ ]:
# Only needed if REPO_PATH is NOT on Drive (e.g. you cloned to /content/)
import shutil

DRIVE_BACKUP = '/content/drive/MyDrive/TezBaselines/MyCode/output'
LOCAL_OUTPUT = str(Path(REPO_PATH) / 'output')

shutil.copytree(LOCAL_OUTPUT, DRIVE_BACKUP, dirs_exist_ok=True)
print(f"Outputs copied to {DRIVE_BACKUP}")